# Random Forest Model Selection

This notebook evaluates a Random Forest classifier with 5-fold stratified cross-validation. Metrics are computed directly from each validation fold.

In [1]:
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, precision_score, f1_score, fbeta_score, average_precision_score, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold

In [2]:
# ==========================================
# 1. Load Training Data ONLY
# ==========================================
train_df = pd.read_csv('../data/processed/train.csv')

X_train = train_df.drop(columns=['Diabetes_01'])
y_train = train_df['Diabetes_01']

In [3]:
# ==========================================
# 2. Setup Stratified Cross-Validation
# ==========================================
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [4]:
# ==========================================
# 3. Build Random Forest Model
# ==========================================
random_forest = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

In [5]:
# ==========================================
# 4. Run 5-Fold Cross-Validation Across Thresholds
# ==========================================

thresholds = [0.5, 0.4, 0.3, 0.25, 0.2]

fold_metrics = []
fold_predictions = []

for fold_number, (train_idx, valid_idx) in enumerate(cv_strategy.split(X_train, y_train), start=1):
    estimator = clone(random_forest)

    X_fold_train = X_train.iloc[train_idx]
    y_fold_train = y_train.iloc[train_idx]
    X_valid = X_train.iloc[valid_idx]
    y_valid = y_train.iloc[valid_idx]

    estimator.fit(X_fold_train, y_fold_train)

    y_valid_proba = estimator.predict_proba(X_valid)[:, 1]

    for threshold in thresholds:
        y_valid_pred = (y_valid_proba >= threshold).astype(int)

        fold_metrics.append({
            "Fold": fold_number,
            "Threshold": threshold,
            "Validation Recall": recall_score(y_valid, y_valid_pred, pos_label=1, zero_division=0),
            "Validation Precision": precision_score(y_valid, y_valid_pred, pos_label=1, zero_division=0),
            "Validation F1": f1_score(y_valid, y_valid_pred, pos_label=1, zero_division=0),
            "Validation F2": fbeta_score(y_valid, y_valid_pred, beta=2, pos_label=1, zero_division=0),
            "Validation AUPRC": average_precision_score(y_valid, y_valid_proba),
            "Validation AUROC": roc_auc_score(y_valid, y_valid_proba),
            "Predicted Positive Rate": y_valid_pred.mean(),
        })

    fold_predictions.append(pd.DataFrame({
        "Fold": fold_number,
        "y_valid": y_valid.to_numpy(),
        "y_valid_proba": y_valid_proba,
    }))

rf_fold_metrics_df = pd.DataFrame(fold_metrics)
rf_cv_predictions_df = pd.concat(fold_predictions, ignore_index=True)

rf_fold_metrics_df.round(6)

,Fold,Threshold,Validation Recall,Validation Precision,Validation F1,Validation F2,Validation AUPRC,Validation AUROC,Predicted Positive Rate
0,1,0.50,0.431879,0.390562,0.410183,0.422931,0.349730,0.782166,0.168883
1,1,0.40,0.588246,0.347831,0.437165,0.516804,0.349730,0.782166,0.258289
2,1,0.30,0.726447,0.303384,0.428017,0.568027,0.349730,0.782166,0.365701
3,1,0.25,0.784328,0.282199,0.415061,0.578469,0.349730,0.782166,0.424480
4,1,0.20,0.840427,0.261890,0.399340,0.582895,0.349730,0.782166,0.490113
5,2,0.50,0.435085,0.398857,0.416184,0.427322,0.361654,0.786543,0.166599
6,2,0.40,0.600178,0.353954,0.445296,0.526875,0.361654,0.786543,0.258969
7,2,0.30,0.732146,0.306745,0.432350,0.573169,0.361654,0.786543,0.364531
8,2,0.25,0.788958,0.283920,0.417570,0.581930,0.361654,0.786543,0.424398
9,2,0.20,0.845592,0.262640,0.400793,0.585623,0.361654,0.786543,0.491718


In [6]:
# ==========================================
# 5. Display Selected Cross-Validation Metrics
# ==========================================

rf_cv_summary_df = (
    rf_fold_metrics_df
    .groupby("Threshold", as_index=False)
    .agg({
        "Validation Recall": "mean",
        "Validation Precision": "mean",
        "Validation F1": "mean",
        "Validation F2": "mean",
        "Validation AUPRC": "mean",
        "Validation AUROC": "mean",
        "Predicted Positive Rate": "mean",
    })
    .rename(columns={
        "Validation Recall": "Validation Recall Mean",
        "Validation Precision": "Validation Precision Mean",
        "Validation F1": "Validation F1 Mean",
        "Validation F2": "Validation F2 Mean",
        "Validation AUPRC": "Validation AUPRC Mean",
        "Validation AUROC": "Validation AUROC Mean",
        "Predicted Positive Rate": "Predicted Positive Rate Mean",
    })
)

default_threshold = 0.5
default_scores = rf_cv_summary_df.loc[
    rf_cv_summary_df["Threshold"] == default_threshold
].copy()
default_scores.insert(0, "Selection Rule", "Default threshold")

best_f2_scores = rf_cv_summary_df.loc[
    [rf_cv_summary_df["Validation F2 Mean"].idxmax()]
].copy()
best_f2_scores.insert(0, "Selection Rule", "Max F2 threshold")

y_valid = rf_cv_predictions_df["y_valid"]
y_valid_proba = rf_cv_predictions_df["y_valid_proba"]
fpr, tpr, roc_thresholds = roc_curve(y_valid, y_valid_proba)
best_tpr_fpr_index = (tpr - fpr).argmax()
best_tpr_fpr_threshold = roc_thresholds[best_tpr_fpr_index]
best_tpr_fpr_pred = (y_valid_proba >= best_tpr_fpr_threshold).astype(int)

best_tpr_fpr_scores = pd.DataFrame([{
    "Selection Rule": "Max TPR-FPR threshold",
    "Threshold": best_tpr_fpr_threshold,
    "Validation Recall Mean": recall_score(y_valid, best_tpr_fpr_pred, pos_label=1, zero_division=0),
    "Validation Precision Mean": precision_score(y_valid, best_tpr_fpr_pred, pos_label=1, zero_division=0),
    "Validation F1 Mean": f1_score(y_valid, best_tpr_fpr_pred, pos_label=1, zero_division=0),
    "Validation F2 Mean": fbeta_score(y_valid, best_tpr_fpr_pred, beta=2, pos_label=1, zero_division=0),
    "Validation AUPRC Mean": average_precision_score(y_valid, y_valid_proba),
    "Validation AUROC Mean": roc_auc_score(y_valid, y_valid_proba),
    "Predicted Positive Rate Mean": best_tpr_fpr_pred.mean(),
    "TPR - FPR": tpr[best_tpr_fpr_index] - fpr[best_tpr_fpr_index],
}])

rf_selected_metrics_df = pd.concat(
    [default_scores, best_f2_scores, best_tpr_fpr_scores],
    ignore_index=True
)

rf_selected_metrics_df.round(6)


,Selection Rule,Threshold,Validation Recall Mean,Validation Precision Mean,Validation F1 Mean,Validation F2 Mean,Validation AUPRC Mean,Validation AUROC Mean,Predicted Positive Rate Mean,TPR - FPR
0,Default threshold,0.500000,0.434519,0.393484,0.412969,0.425630,0.354949,0.784200,0.168667,NaN
1,Max F2 threshold,0.200000,0.843288,0.262804,0.400725,0.584901,0.354949,0.784200,0.490110,NaN
2,Max TPR-FPR threshold,0.267042,0.766143,0.292190,0.423041,0.578477,0.354753,0.784183,0.400492,0.431568
